
## EDA GOLD - Validacion del Star Schema y KPIs
## (pf.gold.* + capa semantica)

In [0]:
%sql
SELECT
  COUNT(*) AS filas_fact,
  COUNT(DISTINCT row_hash) AS row_hash_unicos,
  SUM(
    CASE
      WHEN
        f.fecha_id NOT IN (
          SELECT
            fecha_id
          FROM
            pf.gold.dim_fecha
        )
      THEN
        1
      ELSE 0
    END
  ) AS fecha_huerfana,
  SUM(
    CASE
      WHEN
        f.created_fecha_id NOT IN (
          SELECT
            fecha_id
          FROM
            pf.gold.dim_fecha
        )
      THEN
        1
      ELSE 0
    END
  ) AS created_fecha_huerfana,
  SUM(
    CASE
      WHEN
        f.modelo_id NOT IN (
          SELECT
            modelo_id
          FROM
            pf.gold.dim_modelo_scd2
        )
      THEN
        1
      ELSE 0
    END
  ) AS modelo_huerfano,
  SUM(
    CASE
      WHEN
        f.org_sk NOT IN (
          SELECT
            org_sk
          FROM
            pf.gold.dim_organizacion
        )
      THEN
        1
      ELSE 0
    END
  ) AS org_huerfana,
  SUM(
    CASE
      WHEN
        f.task_id IS NOT NULL
        AND f.task_id NOT IN (
          SELECT
            task_id
          FROM
            pf.gold.dim_task
        )
      THEN
        1
      ELSE 0
    END
  ) AS task_huerfana
FROM
  pf.gold.fact_metricas_diarias f;

In [0]:
%sql
SELECT
  fecha_id,
  COUNT(*) AS filas,
  SUM(downloads) AS descargas_del_dia,
  SUM(delta_downloads) AS incremento_del_dia
FROM
  pf.gold.fact_metricas_diarias
GROUP BY
  fecha_id
ORDER BY
  fecha_id;

In [0]:
%sql
SELECT
  (
    SELECT
      COUNT(DISTINCT model_id)
    FROM
      pf.gold.dim_modelo_scd2
  ) AS modelos_distintos,
  (
    SELECT
      COUNT(*)
    FROM
      pf.gold.dim_modelo_scd2
    WHERE
      is_current = TRUE
  ) AS versiones_vigentes,
  (
    SELECT
      COUNT(*)
        - (
          SELECT
            COUNT(*)
          FROM
            pf.gold.dim_modelo_scd2
          WHERE
            is_current = TRUE
        )
    FROM
      pf.gold.dim_modelo_scd2
  ) AS versiones_historicas,
  (
    SELECT
      COUNT(*)
    FROM
      (
        SELECT
          model_id
        FROM
          pf.gold.dim_modelo_scd2
        WHERE
          is_current = TRUE
        GROUP BY
          model_id
        HAVING
          COUNT(*) <> 1
      )
  ) AS modelos_con_invariante_rota;

In [0]:
%sql
SELECT
  model_id,
  COUNT(*) AS n_versiones
FROM
  pf.gold.dim_modelo_scd2
GROUP BY
  model_id
HAVING
  COUNT(*) > 1
ORDER BY
  n_versiones DESC
LIMIT 10;

In [0]:
%sql
SELECT
  dm.nombre AS modelo,
  dm.org_id AS organizacion,
  df.fecha AS fecha,
  f.downloads AS descargas,
  f.likes AS likes,
  dt.pipeline_tag AS tarea,
  dl.library_name AS libreria,
  dlc.license_tag AS licencia
FROM
  pf.gold.fact_metricas_diarias f
    JOIN pf.gold.dim_fecha df
      ON df.fecha_id = f.fecha_id
    JOIN pf.gold.dim_modelo_scd2 dm
      ON dm.modelo_id = f.modelo_id
      AND dm.is_current = TRUE
    JOIN pf.gold.dim_organizacion do
      ON do.org_sk = f.org_sk
    JOIN pf.gold.dim_task dt
      ON dt.task_id = f.task_id
    JOIN pf.gold.dim_libreria dl
      ON dl.libreria_id = f.libreria_id
    JOIN pf.gold.dim_licencia dlc
      ON dlc.licencia_id = f.licencia_id
ORDER BY
  f.downloads DESC
LIMIT 10;

In [0]:
%sql
SELECT
  df.fecha,
  SUM(f.downloads) AS descargas,
  SUM(f.delta_downloads) AS incremento,
  COUNT(DISTINCT f.model_id) AS modelos_del_dia
FROM
  pf.gold.fact_metricas_diarias f
    JOIN pf.gold.dim_fecha df
      ON df.fecha_id = f.fecha_id
GROUP BY
  df.fecha
ORDER BY
  df.fecha;

In [0]:
%sql
SELECT
  'dim_fecha' AS dim,
  COUNT(*) AS filas
FROM
  pf.gold.dim_fecha
UNION ALL
SELECT
  'dim_organizacion',
  COUNT(*)
FROM
  pf.gold.dim_organizacion
UNION ALL
SELECT
  'dim_task',
  COUNT(*)
FROM
  pf.gold.dim_task
UNION ALL
SELECT
  'dim_libreria',
  COUNT(*)
FROM
  pf.gold.dim_libreria
UNION ALL
SELECT
  'dim_licencia',
  COUNT(*)
FROM
  pf.gold.dim_licencia
UNION ALL
SELECT
  'dim_tag',
  COUNT(*)
FROM
  pf.gold.dim_tag
UNION ALL
SELECT
  'dim_modelo_scd2',
  COUNT(*)
FROM
  pf.gold.dim_modelo_scd2;

In [0]:
%sql
SELECT
  *
FROM
  pf.semantic.vw_kpi_modelos_nuevos
ORDER BY
  fecha_id DESC
LIMIT 10;

In [0]:
%sql
SELECT
  *
FROM
  pf.semantic.vw_kpi_ranking_task
LIMIT 10;

In [0]:
%sql
SELECT
  *
FROM
  pf.semantic.vw_kpi_licencias
LIMIT 10;

In [0]:
%sql
SELECT
  *
FROM
  pf.semantic.vw_kpi_calidad;

In [0]:
%sql
SELECT
  semana,
  descargas_promedio_por_modelo,
  modelos_activos
FROM
  pf.semantic.vw_kpi_adopcion_semanal
ORDER BY
  semana;

In [0]:
%sql
SELECT
  df.mes,
  dt.pipeline_tag AS tarea,
  SUM(f.downloads) AS descargas,
  COUNT(DISTINCT f.model_id) AS modelos
FROM
  pf.gold.fact_metricas_diarias f
    JOIN pf.gold.dim_fecha df
      ON df.fecha_id = f.fecha_id
    JOIN pf.gold.dim_task dt
      ON dt.task_id = f.task_id
GROUP BY
  df.mes,
  dt.pipeline_tag
ORDER BY
  df.mes,
  descargas DESC
LIMIT 30;